# Q-mini-light-0.7M — Human Conversation Assistant (CPU-only)

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lion36508/Q-mini-light-0.7M/blob/main/colab/QMini_Personal_Assistant_CPU_One_Click.ipynb)

**सीधे Google Colab में खोलें — डाउनलोड करने की जरूरत नहीं।** ऊपर badge दबाएँ, फिर Runtime → Run all करें।

यह notebook उसी Q-Mini Light 0.7M architecture और चुने हुए pretrained checkpoint को CPU पर fine-tune करती है। अब synthetic/template examples नहीं बनते। यह OpenAssistant OASST1 के human-generated, human-reviewed conversation data से English examples बनाती है।

- Source: OpenAssistant/oasst1, Apache-2.0
- Filters: English, ready-for-export, reviewed, not deleted, non-synthetic
- Conversation-tree based train/validation split — एक ही conversation दोनों में नहीं जाएगी
- Multi-turn user/assistant context; loss केवल assistant responses पर
- Existing Q-Mini chat tokens and tokenizer are preserved
- Physical batch 4 × gradient accumulation 2 = effective batch 8
- Context 512; CPU only; Drive checkpoints and Q2-v2 export

**Quality gate:** training शुरू होने से पहले tokenizer round-trip और base-model response दिखेगा। अगर base model खुद खराब भाषा पैदा करता है, तो fine-tuning को जादुई इलाज न समझें — वह base checkpoint को ठीक नहीं कर सकती।

In [ ]:
# CPU-only guard — run before importing PyTorch.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("CPU-only mode requested. CUDA will not be used.")

In [ ]:
# Clone the actual repository, install only missing lightweight dependencies, and mount Drive.
from pathlib import Path
import subprocess, sys, importlib.util, json, hashlib, random, time, shutil, math

REPO_URL = "https://github.com/lion36508/Q-mini-light-0.7M.git"
REPO_DIR = Path("/content/Q-mini-light-0.7M")
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO_DIR, check=True)

missing = [name for name in ("tokenizers", "numpy", "datasets") if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing], check=True)

sys.path.insert(0, str(REPO_DIR))
import torch
from qmini.config import QMiniConfig
if torch.cuda.is_available():
    raise RuntimeError("CPU-only guard failed. Select Runtime → Change runtime type → CPU, then restart.")
torch.set_num_threads(4)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

from google.colab import drive
drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_DRIVE = DRIVE_ROOT / "QMini-Light-0.7M"
PROJECT_DRIVE.mkdir(parents=True, exist_ok=True)

repo_config = QMiniConfig.from_json(REPO_DIR / "configs/qmini_light_0.7m.json")
assert repo_config.estimated_parameter_count() == 705742
assert repo_config.max_context == 512
print("Repository:", REPO_URL)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip())
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("Q-mini:", f"{repo_config.estimated_parameter_count():,} parameters | {repo_config.n_experts} experts | Top-{repo_config.top_k}")

In [ ]:
# Find a real trained checkpoint and its matching config/tokenizer in Google Drive.
from qmini.tokenizer import load_tokenizer

NUM_PAIRS = 0  # Set from the downloaded human dataset
OPTIMIZER_STEPS = 1200
PHYSICAL_BATCH = 4
GRAD_ACCUM = 2
EFFECTIVE_BATCH = PHYSICAL_BATCH * GRAD_ACCUM
CONTEXT_LENGTH = 512
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.1
VALID_FRACTION = 0.10
SEED = 42
CPU_THREADS = 4
RUN_NAME = "assistant_sft_oasst1_human_english_v2_endtoken"

SEARCH_ROOTS = [
    PROJECT_DRIVE / "training_ready",
    PROJECT_DRIVE / "training_runs",
    PROJECT_DRIVE / "runs",
    PROJECT_DRIVE / "exports",
    PROJECT_DRIVE / "checkpoints",
]
excluded = {"initial_brain", "initial_brains", "assistant_sft_cpu_runs", "assistant_sft"}
candidates = []
for root in SEARCH_ROOTS:
    if not root.exists():
        continue
    for p in root.rglob("*"):
        if not p.is_file() or any(part.lower() in excluded for part in p.parts):
            continue
        if p.suffix.lower() == ".pt" or (p.suffix.lower() == ".npz" and "final_model" in [x.lower() for x in p.parts]):
            candidates.append(p)
candidates = sorted(set(candidates), key=lambda p: p.stat().st_mtime, reverse=True)

BASE_MODEL_PATH = None
BASE_CONFIG_PATH = None
TOKENIZER_PATH = None
PACKAGE_DIR = None

def sha256_path(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

for candidate in candidates:
    try:
        if candidate.suffix.lower() == ".pt":
            state = torch.load(candidate, map_location="cpu", weights_only=True)
            if not isinstance(state, dict):
                continue
            if state.get("format") == "qmini-checkpoint-v2":
                if int(state.get("step", 0)) <= 0 or state.get("config") is None:
                    continue
                cfg = QMiniConfig.from_dict(state["config"])
            elif "format" not in state and "model" not in state:
                cfg = repo_config
            else:
                continue
        else:
            import numpy as np
            with np.load(candidate, allow_pickle=False) as z:
                meta = json.loads(str(z["metadata_json"].item()))
            if meta.get("format") != "qmini-packed-v2":
                continue
            cfg = QMiniConfig.from_dict(meta["config"])

        if cfg.estimated_parameter_count() != 705742 or cfg.max_context != 512:
            continue

        package = None
        try:
            rel = candidate.relative_to((PROJECT_DRIVE / "training_ready").resolve())
            package = PROJECT_DRIVE / "training_ready" / rel.parts[0]
        except (ValueError, IndexError):
            pass

        config_options = []
        tokenizer_options = []
        if package:
            config_options += [package / "active_config.json", package / "initial_brain" / "config.json"]
            tokenizer_options += [package / "tokenizer" / "tokenizer.json"]
        config_options += [candidate.parent / "config.json", candidate.parent.parent / "config.json", REPO_DIR / "configs/qmini_light_0.7m.json"]
        tokenizer_options += [candidate.parent / "tokenizer.json", candidate.parent.parent / "tokenizer.json"]
        config_path = next((p for p in config_options if p.is_file()), None)
        tok_path = next((p for p in tokenizer_options if p.is_file()), None)
        if config_path is None or tok_path is None:
            continue
        test_cfg = QMiniConfig.from_json(config_path)
        test_tok = load_tokenizer(tok_path)
        if test_cfg.to_dict() != cfg.to_dict() or test_tok.get_vocab_size() != cfg.vocab_size:
            continue
        required = ("<pad>", "<bos>", "<eos>", "<|system|>", "<|user|>", "<|assistant|>", "<|end|>")
        if any(test_tok.token_to_id(t) is None for t in required):
            continue
        BASE_MODEL_PATH, BASE_CONFIG_PATH, TOKENIZER_PATH, PACKAGE_DIR = candidate, config_path, tok_path, package
        break
    except Exception as exc:
        print("Skipping incompatible checkpoint:", candidate, "|", type(exc).__name__, str(exc)[:180])

if BASE_MODEL_PATH is None:
    raise FileNotFoundError(
        "No compatible trained Q-mini checkpoint + matching tokenizer/config were found. "
        "Expected a trained latest.pt under MyDrive/QMini-Light-0.7M/training_ready/<package>/training_state/ "
        "or a Q2-v2 final_model/model.qmini.npz. The random initial_brain is intentionally rejected."
    )

CONFIG = QMiniConfig.from_json(BASE_CONFIG_PATH)
TOKENIZER = load_tokenizer(TOKENIZER_PATH)
assert CONFIG.max_context == CONTEXT_LENGTH and TOKENIZER.get_vocab_size() == CONFIG.vocab_size
print("SELECTED TRAINED MODEL:", BASE_MODEL_PATH)
print("CONFIG:", BASE_CONFIG_PATH)
print("TOKENIZER:", TOKENIZER_PATH)
print("Package:", PACKAGE_DIR)
print("Vocabulary preserved:", CONFIG.vocab_size, "| parameters:", f"{CONFIG.estimated_parameter_count():,}")
print("Effective batch:", EFFECTIVE_BATCH)

In [ ]:
# Build real human-written multi-turn conversation samples from OpenAssistant OASST1.
# No synthetic/template-generated QA records are added.
from datasets import load_dataset
import re

SYSTEM_PROMPT = (
    "You are Q-mini, a friendly, practical and accurate personal AI assistant. "
    "Reply naturally and directly. Answer the user's actual question, ask a short "
    "clarifying question when needed, and do not tell stories unless explicitly asked."
)

OUTPUT_DIR = PROJECT_DRIVE / "assistant_sft_cpu_runs" / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_PATH = OUTPUT_DIR / "qmini_oasst1_human_english.jsonl"

previous_dataset = (
    PROJECT_DRIVE / "assistant_sft_cpu_runs"
    / "assistant_sft_oasst1_human_english_v1"
    / "qmini_oasst1_human_english.jsonl"
)
if not DATASET_PATH.is_file() and previous_dataset.is_file():
    shutil.copy2(previous_dataset, DATASET_PATH)
    print("Reused prior human dataset without re-downloading:", previous_dataset)

if DATASET_PATH.is_file():
    records = [
        json.loads(line)
        for line in DATASET_PATH.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    print("Reused saved OASST1 dataset:", DATASET_PATH)
else:
    ds = load_dataset("OpenAssistant/oasst1")
    by_id = {}

    # Combine the published train/validation messages first, then split by conversation tree below.
    # Filtering keeps only human-written, reviewed English messages in ready-for-export trees.
    for split_name in ds.keys():
        for row in ds[split_name]:
            if row.get("lang") != "en":
                continue
            if row.get("tree_state") != "ready_for_export":
                continue
            if row.get("review_result") is not True:
                continue
            if row.get("deleted") is not False:
                continue
            if row.get("synthetic") is not False:
                continue
            role = row.get("role")
            if role not in ("prompter", "assistant"):
                continue
            text_value = re.sub(r"\s+", " ", str(row.get("text") or "")).strip()
            if not text_value:
                continue
            clean = dict(row)
            clean["text"] = text_value
            by_id[clean["message_id"]] = clean

    records = []
    seen_assistant_ids = set()

    for assistant in by_id.values():
        if assistant["role"] != "assistant":
            continue
        # OASST ranks alternative assistant replies; prefer the top-ranked reply.
        rank = assistant.get("rank")
        if rank is not None and int(rank) != 0:
            continue
        message_id = assistant["message_id"]
        if message_id in seen_assistant_ids:
            continue

        chain = []
        current = assistant
        visited = set()
        complete = True
        while current is not None:
            current_id = current.get("message_id")
            if current_id in visited:
                complete = False
                break
            visited.add(current_id)
            chain.append(current)
            parent_id = current.get("parent_id")
            if not parent_id:
                break
            current = by_id.get(parent_id)
            if current is None:
                complete = False
                break

        if not complete:
            continue
        chain.reverse()
        tree_id = assistant.get("message_tree_id")
        if not tree_id or not chain or chain[0].get("role") != "prompter":
            continue
        if chain[-1].get("role") != "assistant" or len(chain) < 2 or len(chain) % 2:
            continue

        expected_roles = ["prompter" if i % 2 == 0 else "assistant" for i in range(len(chain))]
        if [m["role"] for m in chain] != expected_roles:
            continue
        if any(m.get("message_tree_id") != tree_id for m in chain):
            continue

        # Retain the latest three user/assistant turns for multi-turn context.
        chain = chain[-6:]
        chat_messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        for message in chain:
            chat_messages.append({
                "role": "user" if message["role"] == "prompter" else "assistant",
                "content": message["text"],
            })

        records.append({
            "messages": chat_messages,
            "metadata": {
                "source": "OpenAssistant/oasst1",
                "license": "Apache-2.0",
                "synthetic": False,
                "language": "en",
                "seed_group": tree_id,
                "assistant_message_id": message_id,
            },
        })
        seen_assistant_ids.add(message_id)

    if len(records) < 1000:
        raise RuntimeError(
            f"Only {len(records)} human English conversations survived filtering. "
            "Refusing to train on a tiny or incorrectly parsed dataset."
        )

    random.Random(SEED).shuffle(records)
    with DATASET_PATH.open("w", encoding="utf-8") as f:
        for row in records:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

NUM_PAIRS = len(records)
assert NUM_PAIRS >= 1000
assert all(r["metadata"]["source"] == "OpenAssistant/oasst1" for r in records)
assert all(r["metadata"]["synthetic"] is False for r in records)

tree_count = len({r["metadata"]["seed_group"] for r in records})
print("Dataset:", DATASET_PATH)
print("Human conversation training examples:", NUM_PAIRS)
print("Distinct conversation trees:", tree_count)
print("Synthetic examples:", 0)
print("Source/license: OpenAssistant/oasst1 / Apache-2.0")
print("Language: English (Hinglish/Hindi is not claimed by this English-only subset)")
print("Example roles:", [m["role"] for m in records[0]["messages"]])

## Start CPU-only training

ऊपर का human OASST1 dataset cell पहले चलेगा। उसके बाद नीचे वाला cell वही Q-Mini checkpoint load करके assistant-only fine-tuning करेगा।

In [ ]:
# Build independent chat samples and begin CPU-only instruction fine-tuning.
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from qmini.model import QMiniMoE
from qmini.text import escape_special_token_literals
from qmini.packed_runtime import load_packed_state_into_trainable_model
from qmini.export import export_packed_state, verify_packed_export

# Reload the saved JSONL if the dataset-generation cell was skipped or the Colab runtime restarted.
if "records" not in globals():
    if "DATASET_PATH" not in globals():
        raise RuntimeError(
            "Dataset setup is missing. Run the cells above in order, especially the "
            "'Generate 800 personal-assistant conversations' cell, then rerun this training cell."
        )
    if not Path(DATASET_PATH).is_file():
        raise FileNotFoundError(
            f"Saved chat dataset was not found at {DATASET_PATH}. "
            "Run the dataset-generation cell above first."
        )
    with Path(DATASET_PATH).open("r", encoding="utf-8") as handle:
        records = [json.loads(line) for line in handle if line.strip()]
    print(f"Reloaded {len(records)} conversations from {DATASET_PATH}")

if len(records) != NUM_PAIRS:
    raise ValueError(
        f"Expected {NUM_PAIRS} chat records, found {len(records)}. "
        "Rerun the dataset-generation cell or choose a fresh output folder."
    )

assert PHYSICAL_BATCH == 4 and GRAD_ACCUM == 2 and EFFECTIVE_BATCH == 8
assert CONTEXT_LENGTH == CONFIG.max_context == 512
device = torch.device("cpu")
torch.manual_seed(SEED)
random.seed(SEED)

special = {t: TOKENIZER.token_to_id(t) for t in ("<pad>", "<bos>", "<eos>", "<|system|>", "<|user|>", "<|assistant|>", "<|end|>")}
if any(v is None for v in special.values()):
    raise ValueError("The matching existing tokenizer is missing required Q-mini chat tokens.")
special = {k: int(v) for k, v in special.items()}

groups = {}
for i, row in enumerate(records):
    group = row.get("metadata", {}).get("seed_group", f"record:{i}")
    groups.setdefault(group, []).append(i)
group_names = list(groups)
split_names = list(group_names)
random.Random(SEED).shuffle(split_names)
n_valid = max(1, round(len(group_names) * VALID_FRACTION))
valid_groups = set(split_names[:n_valid])
valid_ids = {i for g in valid_groups for i in groups[g]}
train_records = [r for i, r in enumerate(records) if i not in valid_ids]
valid_records = [r for i, r in enumerate(records) if i in valid_ids]

class ChatDataset(Dataset):
    def __init__(self, rows):
        self.samples = []
        role_tokens = {
            "system": special["<|system|>"],
            "user": special["<|user|>"],
            "assistant": special["<|assistant|>"],
        }
        for row in rows:
            messages = row.get("messages", [])
            system_message = next((m for m in messages if m.get("role") == "system"), None)
            if system_message is None:
                continue
            system_ids = TOKENIZER.encode(
                escape_special_token_literals(system_message.get("content", SYSTEM_PROMPT))
            ).ids[:64]

            conversation = [
                m for m in messages
                if m.get("role") in ("user", "assistant")
            ][-6:]
            if not conversation or conversation[-1].get("role") != "assistant":
                continue
            if conversation[0].get("role") == "assistant":
                conversation = conversation[1:]
            if not conversation or conversation[0].get("role") != "user":
                continue

            encoded = []
            for index, message in enumerate(conversation):
                role = message["role"]
                token_ids = TOKENIZER.encode(
                    escape_special_token_literals(str(message.get("content", "")))
                ).ids
                # Bound each message so even a long OASST reply cannot exceed the 512-token window.
                if len(token_ids) > 192:
                    token_ids = token_ids[:192] if role == "assistant" else token_ids[-192:]
                encoded.append((role, token_ids))

            # Remove oldest complete user/assistant pairs until this real exchange fits.
            def sample_length(items):
                return 1 + (len(system_ids) + 2) + sum(len(tokens) + 2 for _, tokens in items)

            while len(encoded) > 2 and sample_length(encoded) > CONTEXT_LENGTH:
                encoded = encoded[2:]
                if encoded and encoded[0][0] == "assistant":
                    encoded = encoded[1:]
            if not encoded or encoded[-1][0] != "assistant":
                continue
            if sample_length(encoded) > CONTEXT_LENGTH:
                raise ValueError("A human conversation sample still exceeds the 512-token context.")

            ids = [special["<bos>"]]
            mask = [0]

            # System and user tokens are context only; assistant spans receive loss.
            ids += [role_tokens["system"]] + system_ids + [special["<|end|>"]]
            mask += [0] * (len(system_ids) + 2)

            for role, token_ids in encoded:
                ids += [role_tokens[role]] + token_ids + [special["<|end|>"]]
                if role == "assistant":
                    # Supervise response text AND the final <|end|> target.
                    # The following user-role token (if any) remains masked by that user message.
                    mask += [1] * (len(token_ids) + 2)
                else:
                    mask += [0] * (len(token_ids) + 2)

            if len(ids) > CONTEXT_LENGTH or len(ids) != len(mask):
                raise ValueError("Invalid human chat sample length or loss-mask alignment.")
            self.samples.append((ids, mask))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


def collate_chat(batch):
    max_len = max(len(ids) for ids, _ in batch)
    x = torch.full((len(batch), max_len - 1), special["<pad>"], dtype=torch.long)
    y = torch.full((len(batch), max_len - 1), -100, dtype=torch.long)
    for i, (ids_list, mask_list) in enumerate(batch):
        ids = torch.tensor(ids_list, dtype=torch.long)
        mask = torch.tensor(mask_list[1:], dtype=torch.bool)
        n = len(ids) - 1
        x[i, :n] = ids[:-1]
        targets = ids[1:]
        y[i, :n] = torch.where(mask, targets, torch.full_like(targets, -100))
    return {"input_ids": x, "labels": y}

train_data = ChatDataset(train_records)
valid_data = ChatDataset(valid_records)
if len(train_data) < 8 or not len(valid_data):
    raise ValueError("Train/validation split is too small.")

# Guard against a regression in assistant end-marker supervision.
for split_name, dataset_obj in (("train", train_data), ("validation", valid_data)):
    for sample_ids, sample_mask in dataset_obj.samples[:min(32, len(dataset_obj))]:
        if sample_ids[-1] != special["<|end|>"] or sample_mask[-1] != 1:
            raise RuntimeError(
                f"{split_name} sample does not supervise the assistant <|end|> token. "
                "Refusing to train without an explicit assistant turn-ending target."
            )
print("Assistant end-token supervision: PASS")

loader_gen = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_data, batch_size=PHYSICAL_BATCH, shuffle=True, drop_last=True, num_workers=0, collate_fn=collate_chat, generator=loader_gen)
valid_loader = DataLoader(valid_data, batch_size=PHYSICAL_BATCH, shuffle=False, drop_last=False, num_workers=0, collate_fn=collate_chat)

model = QMiniMoE(CONFIG).to(device)
if BASE_MODEL_PATH.suffix.lower() == ".npz":
    load_packed_state_into_trainable_model(model, BASE_MODEL_PATH)
else:
    base = torch.load(BASE_MODEL_PATH, map_location="cpu", weights_only=True)
    if base.get("format") == "qmini-checkpoint-v2":
        if base.get("config") != CONFIG.to_dict():
            raise ValueError("Pretraining checkpoint config does not match selected config.")
        model.load_state_dict(base["model"], strict=True)
    elif "format" not in base and "model" not in base:
        model.load_state_dict(base, strict=True)
    else:
        raise ValueError("Unsupported checkpoint format. Select a Q-mini checkpoint or Q2-v2 final model.")

# Sanity-check the tokenizer and show what the base checkpoint can already say before SFT.
from qmini.generate import generate
probe_text = "Hello, how are you?"
probe_round_trip = TOKENIZER.decode(TOKENIZER.encode(probe_text).ids)
if probe_round_trip != probe_text:
    raise ValueError(
        "Tokenizer round-trip failed. The selected tokenizer may not belong to this checkpoint; "
        "do not fine-tune or overwrite anything until the matching tokenizer is located."
    )
print("Tokenizer round-trip: PASS")
base_prompt = (
    "<|system|>You are a helpful assistant.<|end|>"
    "<|user|>Hi<|end|><|assistant|>"
)
model.eval()
base_preview = generate(
    model, TOKENIZER, base_prompt,
    max_new_tokens=32, temperature=0.0, top_k=1, top_p=1.0,
    thinking_steps=CONFIG.thinking_steps, serialized_prompt=True
)
print("BASE MODEL BEFORE HUMAN SFT:", base_preview)
print("This is only a base-language readability check; it is NOT an assistant-quality test.")
print("A story-like continuation can be normal before instruction fine-tuning.")
print("Type YES only if the English words are readable; type NO if the text is corrupted/gibberish.")
base_ok = input("Continue to human-data fine-tuning? Type YES/NO: ").strip().upper()
if base_ok != "YES":
    raise RuntimeError(
        "Stopped before fine-tuning. The selected base checkpoint must first pass the language-generation check."
    )
model.train()

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY)
base_sha, data_sha, tok_sha = sha256_path(BASE_MODEL_PATH), sha256_path(DATASET_PATH), sha256_path(TOKENIZER_PATH)
checkpoint_path = OUTPUT_DIR / "assistant_sft_latest.pt"
metrics_path = OUTPUT_DIR / "assistant_sft_metrics.jsonl"
start_step = 0
if checkpoint_path.is_file():
    saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    expected = {"format": "qmini-chat-sft-v1", "base_sha": base_sha, "data_sha": data_sha, "tokenizer_sha": tok_sha, "config": CONFIG.to_dict()}
    for k, v in expected.items():
        if saved.get(k) != v:
            raise ValueError(f"Existing SFT checkpoint mismatch ({k}). Choose a new RUN_NAME instead of overwriting.")
    model.load_state_dict(saved["model"], strict=True)
    optimizer.load_state_dict(saved["optimizer"])
    start_step = int(saved["step"])
    print("Resuming existing SFT run at step", start_step)

def evaluate():
    model.eval()
    loss_sum, targets = 0.0, 0
    with torch.inference_mode():
        for bi, batch in enumerate(valid_loader):
            if bi >= 10:
                break
            out = model(batch["input_ids"], thinking_steps=CONFIG.thinking_steps)
            labels = batch["labels"]
            n = int((labels != -100).sum())
            if n:
                loss_sum += float(F.cross_entropy(out["logits"].float().reshape(-1, CONFIG.vocab_size), labels.reshape(-1), ignore_index=-100, reduction="sum"))
                targets += n
    model.train()
    if not targets:
        raise ValueError("Validation split has no assistant targets.")
    return loss_sum / targets

def save_checkpoint(step):
    payload = {
        "format": "qmini-chat-sft-v1", "step": step, "model": model.state_dict(),
        "optimizer": optimizer.state_dict(), "config": CONFIG.to_dict(),
        "base_sha": base_sha, "data_sha": data_sha, "tokenizer_sha": tok_sha,
        "torch_rng": torch.get_rng_state(), "loader_rng": loader_gen.get_state()
    }
    tmp = checkpoint_path.with_suffix(".tmp")
    torch.save(payload, tmp)
    os.replace(tmp, checkpoint_path)

if start_step > 0 and checkpoint_path.is_file():
    torch.set_rng_state(saved["torch_rng"])
    loader_gen.set_state(saved["loader_rng"])
iterator = iter(train_loader)

model.train()
for step in range(start_step, OPTIMIZER_STEPS):
    optimizer.zero_grad(set_to_none=True)
    micro = []
    for _ in range(GRAD_ACCUM):
        try:
            micro.append(next(iterator))
        except StopIteration:
            iterator = iter(train_loader)
            micro.append(next(iterator))
    total_targets = sum(int((b["labels"] != -100).sum()) for b in micro)
    if total_targets <= 0:
        raise ValueError("No assistant targets in effective batch.")
    ce_sum_total = 0.0
    aux_total = 0.0
    for batch in micro:
        out = model(batch["input_ids"], thinking_steps=CONFIG.thinking_steps)
        ce_sum = F.cross_entropy(out["logits"].float().reshape(-1, CONFIG.vocab_size), batch["labels"].reshape(-1), ignore_index=-100, reduction="sum")
        ce = ce_sum / total_targets
        aux = out["aux_loss"].float()
        loss = ce + CONFIG.router_aux_loss_coef * aux / GRAD_ACCUM
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite loss at step {step + 1}")
        loss.backward()
        ce_sum_total += float(ce_sum.detach())
        aux_total += float(aux.detach()) / GRAD_ACCUM
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
    warmup = max(1, int(OPTIMIZER_STEPS * 0.03))
    if step < warmup:
        lr = LEARNING_RATE * (step + 1) / warmup
    else:
        prog = min(1.0, (step - warmup) / max(1, OPTIMIZER_STEPS - warmup))
        lr = LEARNING_RATE * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * prog)))
    for group in optimizer.param_groups:
        group["lr"] = lr
    optimizer.step()
    completed = step + 1
    train_ce = ce_sum_total / total_targets
    valid_ce = evaluate() if completed % 25 == 0 or completed == OPTIMIZER_STEPS else None
    rec = {"step": completed, "train_assistant_ce": train_ce, "validation_assistant_ce": valid_ce, "assistant_target_tokens": total_targets, "router_aux_loss": aux_total, "gradient_norm": float(grad_norm), "learning_rate": lr, "device": "cpu"}
    with metrics_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(rec) + "\n")
    if completed % 5 == 0 or completed == 1 or completed == OPTIMIZER_STEPS:
        print(f"step {completed}/{OPTIMIZER_STEPS} | assistant CE={train_ce:.4f} | validation CE={valid_ce if valid_ce is not None else 'pending'}")
    if completed % 25 == 0 or completed == OPTIMIZER_STEPS:
        save_checkpoint(completed)

model.eval()
torch.save(model.state_dict(), OUTPUT_DIR / "assistant_sft_model.pt")
CONFIG.save(OUTPUT_DIR / "config.json")
shutil.copy2(TOKENIZER_PATH, OUTPUT_DIR / "tokenizer.json")
packed_path = OUTPUT_DIR / "model.qmini.npz"
export_packed_state(model.state_dict(), CONFIG, packed_path)
packed_report = verify_packed_export(packed_path)
assert packed_report.get("format") == "qmini-packed-v2"
manifest = {
    "task": "personal_assistant_instruction_sft",
    "device": "cpu",
    "parameters": sum(p.numel() for p in model.parameters()),
    "physical_batch": PHYSICAL_BATCH,
    "gradient_accumulation": GRAD_ACCUM,
    "effective_batch": EFFECTIVE_BATCH,
    "context_length": CONTEXT_LENGTH,
    "steps": OPTIMIZER_STEPS,
    "dataset_pairs": len(records),
    "dataset_source": "OpenAssistant/oasst1",
    "dataset_license": "Apache-2.0",
    "human_generated_non_synthetic": True,
    "train_pairs": len(train_data),
    "validation_pairs": len(valid_data),
    "base_model_sha256": base_sha,
    "dataset_sha256": data_sha,
    "tokenizer_sha256": tok_sha,
    "packed_format": "qmini-packed-v2",
    "synthetic_examples": 0
}
(OUTPUT_DIR / "assistant_sft_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print("\nSFT export complete:", packed_path)
print("Base weights:", BASE_MODEL_PATH)
print("Output directory:", OUTPUT_DIR)

In [ ]:
# View the assistant training curve.
import pandas as pd
import matplotlib.pyplot as plt
metric_lines = [json.loads(x) for x in metrics_path.read_text(encoding="utf-8").splitlines() if x.strip()]
df = pd.DataFrame(metric_lines)
plt.figure(figsize=(10,4))
plt.plot(df["step"], df["train_assistant_ce"], label="Training assistant CE")
valid = df.dropna(subset=["validation_assistant_ce"])
if len(valid):
    plt.plot(valid["step"], valid["validation_assistant_ce"], marker="o", label="Validation assistant CE")
plt.xlabel("Optimizer step")
plt.ylabel("Cross-entropy loss")
plt.title("Q-mini Personal Assistant — CPU SFT")
plt.grid(True, alpha=0.25)
plt.legend()
plt.show()
display(df.tail(10))

In [ ]:
# Compare the trained QAT model with its packed Q2-v2 deployment export.
from qmini.generate import load_model, generate
assistant_model, assistant_tokenizer, loaded_device = load_model(str(OUTPUT_DIR), device="cpu")
assistant_model.eval()
model.eval()

# Match the system instruction used by the human-data training examples.
SYSTEM = SYSTEM_PROMPT
QUESTIONS = [
    "Hi",
    "How are you?",
    "What are you doing?",
    "Can you explain what a transformer is in one simple sentence?",
]

def make_prompt(question):
    return (
        f"<|system|>{SYSTEM}<|end|>"
        f"<|user|>{question}<|end|>"
        f"<|assistant|>"
    )

def run_generation(candidate, candidate_tokenizer, question, seed):
    torch.manual_seed(seed)
    answer = generate(
        candidate,
        candidate_tokenizer,
        make_prompt(question),
        max_new_tokens=64,
        temperature=0.7,
        top_k=40,
        top_p=0.9,
        thinking_steps=candidate.config.thinking_steps,
        serialized_prompt=True,
        frequency_penalty=0.25,
        presence_penalty=0.1,
    )
    token_ids = candidate_tokenizer.encode(answer).ids if answer else []
    unique_ratio = len(set(token_ids)) / max(1, len(token_ids))
    return {"answer": answer, "tokens": len(token_ids), "unique_token_ratio": unique_ratio}

evaluation = []
print("Loaded device:", loaded_device)
for index, question in enumerate(QUESTIONS):
    qat = run_generation(model, TOKENIZER, question, 1000 + index)
    q2 = run_generation(assistant_model, assistant_tokenizer, question, 1000 + index)
    evaluation.append({
        "question": question,
        "qat_model": qat,
        "packed_q2_model": q2,
    })
    print("\nUSER:", question)
    print("QAT FP32-master / fake-Q2 path:", qat["answer"])
    print("Packed Q2-v2 path:", q2["answer"])
    print("Q2 unique-token ratio:", round(q2["unique_token_ratio"], 3))

report_path = OUTPUT_DIR / "assistant_sft_generation_evaluation.json"
report_path.write_text(json.dumps(evaluation, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("\nGeneration comparison saved:", report_path)
print("Saved files:", [p.name for p in OUTPUT_DIR.iterdir() if p.is_file()])